# Dozer (machine 5): track final-drive gear wear — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | CORA vibration (Vx, 3 kHz) + T1 temperature | CORA infrared camera |
| Classes | **healthy, worn** (gear wear 25/50/75% together) | same two |
| Tested on | 16 runs, one speed held out at a time | the same 16 runs |
| Code | `Models/dozer.py csv` | `Models/dozer.py cnn-pretrained` |

**Important:** there is no public dozer dataset with matching sensor and camera data. As planned in the dataset manifest, the Dozer final drive uses the **same 16 CORA gear-wear runs as the Belt Conveyor (not new data)** with the Dozer's own target: healthy vs gear-worn. Only 4 of the 16 runs are healthy, so always guessing "worn" already scores 0.750: read **balanced accuracy** and **macro F1**. Lab data, not a dozer; it classifies the present condition, not when to repair.

All of this machine's experiment code is in `Models/dozer.py` (data loading shared with `Models/belt_conveyor.py`); shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [1]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: CORA T1 samples", "Processed_Data/cora_gear_wear/t1_bounded_samples.npz", "*", 1),
    ("Sensor: CORA vibration (Vx)", "Raw_Data/cora_vibration", "*_Vx.csv", 16),
    ("Camera: CORA infrared stacks", "Raw_Data/cora_gear_wear", "*.mat", 16),
])

,environment
git commit,0bdc3b3
python,3.11.9
kernel,project .venv
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: CORA T1 samples,Processed_Data/cora_gear_wear/t1_bounded_sampl...,1/1,1.7,True
Sensor: CORA vibration (Vx),Raw_Data/cora_vibration,28/16,3078.1,True
Camera: CORA infrared stacks,Raw_Data/cora_gear_wear,16/16,390.5,True


All data is local: running fully offline.


## 1. Sensor: vibration and temperature (CSV)
Each run's vibration becomes 1-second windows measured in **shaft orders** (multiples of the shaft speed), optionally with the run's T1 temperature. Gear-tooth wear shows at higher orders and in spikes. The features go into a CSV table; the model and feature set are chosen on validation speeds inside the training data, then tested once on the held-out speed.

In [2]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "csv", tail=2)                    # main: unseen speed
    run_script("Models/dozer.py", "csv", "--split", "time", tail=1)  # secondary: time split
csv_result, time_result = load("dozer_csv_2class.json"), load("dozer_csv_2class_timesplit.json")
show({"MAIN (unseen speed) | Sensor: validation-selected": csv_result["validation_selected"]["score"],
      **{f"MAIN (unseen speed) | Sensor: {k}": v for k, v in csv_result["models"].items()},
      **{f"TIME SPLIT | Sensor: {k}": v for k, v in time_result["models"].items()}})

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[Models/dozer.py csv]
  Validation-selected model: accuracy 0.625  macro F1 0.564  (choices: ['vibration / Extra trees', 'vibration+temperature (per-speed normalised) / Gradient boosting', 'vibration+temperature (per-speed normalised) / Extra trees', 'vibration+temperature (per-speed normalised) / Logistic regression'])
  Saved Processed_Data\dozer_sensor_features_2class.csv, Results\dozer\dozer_csv_2class.json and Figures\dozer\dozer_csv_2class.png


[Models/dozer.py csv --split time]
  Saved Results\dozer\dozer_csv_2class_timesplit.json and Figures\dozer\dozer_csv_2class_timesplit.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
MAIN (unseen speed) | Sensor: validation-selected,0.625,0.583,0.567,0.583,0.564
MAIN (unseen speed) | Sensor: Majority class (control),0.750,0.500,0.375,0.500,0.429
MAIN (unseen speed) | Sensor: Logistic regression,0.562,0.625,0.595,0.625,0.547
MAIN (unseen speed) | Sensor: Random forest,0.812,0.625,0.900,0.625,0.644
TIME SPLIT | Sensor: Majority class (control),0.750,0.500,0.375,0.500,0.429
TIME SPLIT | Sensor: Logistic regression,0.875,0.917,0.833,0.917,0.855
TIME SPLIT | Sensor: Random forest,1.000,1.000,1.000,1.000,1.000


![CSV sensor model](../Figures/dozer/dozer_csv_2class.png) ![Time split](../Figures/dozer/dozer_csv_2class_timesplit.png)

**Two tests, clearly separated.** *Main (unseen speed):* each whole speed is held out (1 healthy + 3 worn runs), so the model must recognise wear at a speed it never trained on. *Time split (secondary, easier):* the first 60% of every 50-minute run trains, the last 30% tests (10% gap); the same recordings appear in training, so part of a high score may come from recognising the recording rather than the wear.

## 2. Camera: infrared images with a pretrained CNN
`export-images` saves 150 infrared frames per run as `Images/dozer/<class>/dozer_<class>_<wear level>_<speed>_fNNN.png`. A pretrained ResNet18 is fine-tuned with data augmentation; the number of training rounds is chosen on validation runs. Retraining takes about half an hour, so it is off by default.

In [3]:
TRAIN_CNN = False
if TRAIN_CNN and all(LOCAL.values()):
    run_script("Models/dozer.py", "export-images", tail=1)
    run_script("Models/dozer.py", "cnn-pretrained", tail=1)
    run_script("Models/dozer.py", "cnn-pretrained", "--split", "time", tail=1)
rows = {}
for name, label in (("dozer_cnn_pretrained_2class.json", "MAIN (unseen speed) | Camera: pretrained ResNet18"),
                    ("dozer_cnn_pretrained_2class_timesplit.json", "TIME SPLIT | Camera: pretrained ResNet18")):
    r = load(name)
    if r:
        rows[f"{label} (per run)"] = r["recording_level"]
show(rows)

,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
MAIN (unseen speed) | Camera: pretrained ResNet18 (per run),0.75,0.5,0.375,0.5,0.429
TIME SPLIT | Camera: pretrained ResNet18 (per run),1.00,1.0,1.000,1.0,1.000


![Pretrained CNN](../Figures/dozer/dozer_cnn_pretrained_2class.png) ![Time split](../Figures/dozer/dozer_cnn_pretrained_2class_timesplit.png)

**Data (CC BY 4.0):** CORA v2.1, DOI 10.34810/DATA2500, the same files as the Belt Conveyor.

## Proposal Week 6: advanced ensemble models (XGBoost / LightGBM)
The same CSV sensor features and the same held-out recordings, now with **XGBoost** and **LightGBM** next to random forest, extra trees and scikit-learn gradient boosting. Settings are fixed in advance; nothing is tuned on the test. `train_seconds` is the total training time.

In [4]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "ensembles", tail=1)
    run_script("Models/dozer.py", "ensembles", "--split", "time", tail=1)
show_ensembles("dozer")

[Models/dozer.py ensembles]
  Saved Results\dozer\dozer_ensembles.json and Figures\dozer\dozer_ensembles.png


[Models/dozer.py ensembles --split time]
  Saved Results\dozer\dozer_ensembles_timesplit.json and Figures\dozer\dozer_ensembles_timesplit.png


,accuracy,balanced_accuracy,macro_f1,train_seconds
MAIN | Majority class (control),0.750,0.500,0.429,0.00
MAIN | Random forest,0.812,0.625,0.644,3.40
MAIN | Extra trees,0.750,0.750,0.709,1.72
MAIN | Gradient boosting (scikit-learn),0.688,0.542,0.543,8.15
MAIN | XGBoost,0.812,0.625,0.644,0.97
MAIN | LightGBM,0.750,0.583,0.590,1.11
TIME SPLIT (secondary) | Majority class (control),0.750,0.500,0.429,0.00
TIME SPLIT (secondary) | Random forest,1.000,1.000,1.000,0.66
TIME SPLIT (secondary) | Extra trees,1.000,1.000,1.000,0.38
TIME SPLIT (secondary) | Gradient boosting (scikit-learn),1.000,1.000,1.000,1.60


![Ensemble comparison](../Figures/dozer/dozer_ensembles.png)

## Week 7: anomaly detection (trained on healthy data only)
Isolation Forest and One-Class SVM learn only what **healthy** looks like; anything unusual raises an alarm. No fault labels are used for training. The alarm threshold is set from the healthy training recordings before testing. **ROC-AUC** (1.0 = perfect, 0.5 = guessing) shows how well the score separates healthy from faulty; **fault detection rate** and **false alarm rate** are at that fixed threshold.

In [5]:
if all(LOCAL.values()):
    run_script("Models/dozer.py", "anomaly", tail=1)
    run_script("Models/dozer.py", "anomaly", "--split", "time", tail=1)
show_anomaly("dozer")

[Models/dozer.py anomaly]
  Saved Results\dozer\dozer_anomaly.json and Figures\dozer\dozer_anomaly.png


[Models/dozer.py anomaly --split time]
  Saved Results\dozer\dozer_anomaly_timesplit.json and Figures\dozer\dozer_anomaly_timesplit.png


,roc_auc,fault_detection_rate,false_alarm_rate,accuracy,test_recordings
MAIN | Isolation Forest,0.6250,0.0000,0.00,0.2500,16.0
MAIN | One-Class SVM,0.5833,0.3333,0.25,0.4375,16.0
TIME SPLIT (secondary) | Isolation Forest,0.9167,0.0000,0.00,0.2500,16.0
TIME SPLIT (secondary) | One-Class SVM,0.7708,0.5000,0.00,0.6250,16.0


![Anomaly scores](../Figures/dozer/dozer_anomaly.png)

Each dot is one test recording; above the dashed line = alarm. Green = healthy, red = faulty.

## Try your own CSV
Put the path of any CSV in `CSV_FILE` and run this cell. It can be a **feature table** (same columns as `Processed_Data/dozer_sensor_features_2class.csv`, optional `class` column) or a **raw recording** (one column of CORA Vx vibration values). Set `LABEL` to the true class to see the accuracy. Classes: healthy, worn. The saved model is trained on all project data, so testing on one of the project's own recordings gives an optimistic score; to test fairly, first run `python Models/dozer.py train-final --exclude <recording>`.

In [6]:
CSV_FILE = None   # e.g. r"C:/Users/me/Downloads/my_recording.csv"
LABEL = None      # true class if known
SPEED = None      # raw recordings only: "F5", "F15", "F50" or "F60"
if CSV_FILE:
    run_script("Models/dozer.py", "predict", "--csv", CSV_FILE, *((["--label", LABEL] if LABEL else []) + (["--speed", SPEED] if SPEED else [])), tail=25)
else:
    print("Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).")

Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).
